In [0]:
# Step 1 - Read Parquet
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark.sql("USE CATALOG Bronze_Catalog")
spark.sql("USE SCHEMA Bronze_SCH")



In [0]:
grid_df = spark.read.parquet(
    "abfss://pro2@adlsstgacntp08.dfs.core.windows.net/parquet/grid_load_stream_v2.parquet"
)

In [0]:
# Step 2 - Check Schema
grid_df.printSchema()


In [0]:
# Step 1 - Add Technical Metadata Column
from pyspark.sql.functions import current_timestamp

grid_df = grid_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

In [0]:
# Step 2 - Verify
grid_df.printSchema()

In [0]:
print("Total Records :", grid_df.count())

In [0]:

# Read Watermark
last_watermark = spark.sql("""
SELECT last_processed_timestamp
FROM Bronze_Catalog.Bronze_SCH.Watermark_Table
WHERE table_name='Bronze_Grid_Load'
""").collect()[0][0]

print(last_watermark)

In [0]:
spark.sql("""
INSERT INTO Bronze_Catalog.Bronze_SCH.Watermark_Table
VALUES (
'Bronze_Grid_Load',
TIMESTAMP('1900-01-01 00:00:00')
);
""")

In [0]:
# Incremental Filter
incremental_df = grid_df.filter(
    col("ingestion_timestamp") > last_watermark
)

records_loaded = incremental_df.count()

print(f"New Records : {records_loaded}")

display(incremental_df.limit(10))

In [0]:
from pyspark.sql.functions import current_timestamp

try:

    incremental_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema","true") \
        .saveAsTable("Bronze_Catalog.Bronze_SCH.Bronze_Grid_Load")

    spark.sql(f"""
        INSERT INTO Bronze_Catalog.Bronze_SCH.Audit_Log
        VALUES(
            'Bronze_Grid_Load',
            'Grid Bronze Pipeline',
            current_timestamp(),
            {records_loaded},
            'SUCCESS',
            NULL
        )
    """)

    print(f"✅ Bronze_Grid_Load loaded successfully.")
    print(f"📊 Records Loaded : {records_loaded}")
    
except Exception as e:

    error_message = str(e).replace("'", "''")

    spark.sql(f"""
        INSERT INTO Bronze_Catalog.Bronze_SCH.Audit_Log
        VALUES(
            'Bronze_Grid_Load',
            'Grid Bronze Pipeline',
            current_timestamp(),
            0,
            'FAILED',
            '{error_message}'
        )
    """)

    print("❌ Bronze_Grid_Load Pipeline Failed")
    print(f"⚠️ Error : {error_message}")
    

    raise

In [0]:

# Update Watermark
spark.sql("""
UPDATE Bronze_Catalog.Bronze_SCH.Watermark_Table
SET last_processed_timestamp = (
    SELECT MAX(ingestion_timestamp)
    FROM Bronze_Catalog.Bronze_SCH.Bronze_Grid_Load
)
WHERE table_name='Bronze_Grid_Load'
""")

print("✅ Watermark Updated Successfully")

In [0]:
# Verify Watermark
spark.sql("""
SELECT *
FROM Bronze_Catalog.Bronze_SCH.Watermark_Table
WHERE table_name='Bronze_Grid_Load'
""").show(truncate=False)

In [0]:
spark.sql("""
SELECT *
FROM Bronze_Catalog.Bronze_SCH.Audit_Log
WHERE table_name='Bronze_Grid_Load'
ORDER BY load_time DESC
LIMIT 5
""").show(truncate=False)